In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(".env", override=True)
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"


In [3]:
def get_completion_from_messages(
    messages,
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_tokens=500
):
    system_instruction = ""
    contents = []

    for message in messages:
        if message["role"] == "system":
            system_instruction = message["content"]

        elif message["role"] == "user":
            contents.append({
                "role": "user",
                "parts": [{"text": message["content"]}]
            })

        elif message["role"] == "assistant":
            contents.append({
                "role": "model",
                "parts": [{"text": message["content"]}]
            })

    response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    return response.text

In [4]:
delimiter = "####"
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}. 

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count. 

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products: 
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter}: If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information. 

Step 5:{delimiter}: First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""

In [5]:
user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 

response = get_completion_from_messages(messages)
print(response)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Step 1:#### The user is asking a question comparing the prices of two specific products: the BlueWave Chromebook and the TechPro Desktop.
Step 2:#### Both products are in the provided list: the BlueWave Chromebook (Model: BW-CB100, Price: $249.99) and the TechPro Desktop (Model: TP-DT500, Price: $999.99).
Step 3:#### The user is assuming that the BlueWave Chromebook is more expensive than the TechPro Desktop.
Step 4:#### The assumption is false. The BlueWave Chromebook costs $249.99, while the TechPro Desktop costs $999.99, making the desktop more expensive than the Chromebook.
Response to user:#### Hello! Actually, you have it the other way around. The BlueWave Chromebook is priced at $249.99, while the TechPro Desktop is more expensive at $999.99. Therefore, the TechPro Desktop is actually $750.00 more expensive than the BlueWave Chromebook! Let me know if you need help with anything else.


In [6]:
user_message = f"""
do you sell tvs"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 
response = get_completion_from_messages(messages)
print(response)

Step 1:#### The user is asking about a product category ("tvs") rather than a specific product.
Step 2:#### N/A as the user did not ask about specific products.
Step 3:#### N/A as no specific products or assumptions about them were mentioned in relation to the available list.
Step 4:#### N/A as there are no assumptions to evaluate.
Response to user:#### Hello! No, we do not sell TVs. We specialize in computers and laptops, such as the TechPro Ultrabook, BlueWave Gaming Laptop, PowerLite Convertible, TechPro Desktop, and BlueWave Chromebook. Let me know if you would like to know more about any of our computers or laptops!


In [7]:
try:
    final_response = response.split(delimiter)[-1].strip()
except Exception as e:
    final_response = "Sorry, I'm having trouble right now, please try asking another question."
    
print(final_response)

Hello! No, we do not sell TVs. We specialize in computers and laptops, such as the TechPro Ultrabook, BlueWave Gaming Laptop, PowerLite Convertible, TechPro Desktop, and BlueWave Chromebook. Let me know if you would like to know more about any of our computers or laptops!
